In [ ]:
!pip install lightgbm optuna koreanize-matplotlib --quiet


In [ ]:
# day1 _수정코드_LGBM결과변함없음.ipynb

In [ ]:
"""
==============================================================================
 난임 환자 임신 성공 예측 해커톤 - 실험 기록
 데이터: 256,351행 / 69열 / 타깃: 임신 성공 여부 (성공 25.8% : 실패 74.2%)
 평가지표: ROC-AUC / 검증방식: Stratified 5-Fold OOF
==============================================================================

 버전            | OOF AUC | 리더보드 | 주요 변경사항
----------------|---------|----------|----------------------------------
 베이스라인      | 0.7392  |    -     | 원본 피처 그대로 사용
                |         |          | scale_pos_weight=2.87 설정
                |         |          | LightGBM 기본 파라미터
----------------|---------|----------|----------------------------------
 v2 (중복 실패) | 0.7390  |    -     | 파생 피처 추가했으나 원본과 중복
                |         |          | 나이_수치 + 시술 당시 나이 동시 투입
                |         |          | → colsample에서 서로 희석되어 하락
                |         |          | 교훈: 파생 피처 만들면 원본 제거
----------------|---------|----------|----------------------------------
 v3 (팀통합)    | 0.7395  | 0.74119  | 팀원 나이 피처 채택
                |         |          | Age_Median: 실제 중앙값(26~47.5)
                |         |          | Age_Risk_Group: Normal/High_Early/
                |         |          |   High_Extreme 3단계 임상 위험군
                |         |          | 시술 당시 나이(카테고리) 제거
----------------|---------|----------|----------------------------------
 v4 (기증난자)  | 0.7394  |    -     | 기증난자×나이 교호작용 추가 시도
                |         |          | ** '기증' → '기증 제공'
                |         |          | 수정 후에도 AUC 동일
                |         |          | 원인: LightGBM이 난자 출처 컬럼으로
                |         |          |   이미 해당 패턴 학습 중이었음
----------------|---------|----------|----------------------------------
 v4 + Optuna   | 0.7400  |    -     | Optuna 50회 탐색 (약 60분 소요)
                |         |          | 핵심: num_leaves 127→32 (단순화)
                |         |          | colsample_bytree 0.7→0.466
                |         |          | reg_alpha 0.1→3.546 (정규화 강화)
                |         |          | best_iter 169~290 → 238~409
----------------|---------|----------|----------------------------------
 v4 + CatBoost | 0.7402  |    -     | CatBoost 단독 모델 추가
                |         |          | 범주형을 Target Statistics로 처리
                |         |          | iterations=2000, depth=6
                |         |          | LightGBM보다 소폭 높은 AUC
----------------|---------|----------|----------------------------------
 앙상블 (최종)  | 0.7404  |    -     | LGBM Optuna + CatBoost
                |         |          | 자동 가중치 0.500 : 0.500
                |         |          | (두 모델 AUC 거의 동일하여 균등)
                |         |          | 단일 모델 대비 +0.0002 추가 개선
==============================================================================

 [삭제한 변수 - 팀 결정]
  - ID: 식별자
  - 임신 시도 또는 마지막 임신 경과 연수: 결측 96%
  - 난자 해동 경과일: 결측 96%
  - PGS/PGD 시술 여부, 착상 전 유전 검사: 결측 99% → 유전검사_시행여부로 대체
  - 불임 원인 정자 계열 6개 + 여성/자궁경부: 99% 이상이 0인 희소 변수

 [생성한 파생 변수]
  - Age_Median        : 나이 구간 → 실제 중앙값 (팀원 코드)
  - Age_Risk_Group    : 임상 위험군 3단계 분류 (팀원 코드)
  - 유전검사_시행여부  : PGS 또는 PGD 시행 여부 이진화
  - 배아_잉여율        : (총생성 - 이식) / 총생성
  - 수정률            : 혼합 난자 수 / 수집 신선 난자 수
  - 저장배아_보유여부  : 저장 배아 > 0 이진화
  - 총_시술횟수        : IVF + DI 횟수 합산
  - 기증정자_사용여부  : 기증자 정자 혼합 난자 수 결측 여부
  - 신선주기_여부      : 배아 해동 경과일 결측 여부
  - Age×배아수        : Age_Median × 총 생성 배아 수 교호작용
  - 목적_* (5개)      : 배아 생성 주요 이유 멀티라벨 분리

 [핵심 발견]
  - 배아 이식 경과일: 누수 아님 확정 (Day5 성공률 40.4% - 이식 전 결정)
  - 이식없음 케이스: 성공률 2.4% (이식있음 30.6%와 12배 차이)
  - 단일 배아 이식 역설: 선택 편향 (우수 배아에만 SET 적용)
  - 만45-50세 역전: 기증 난자 사용 비율 증가가 원인 (기증 31.5% vs 본인 25.8%)
  - num_leaves=32가 127보다 우수: 이 데이터는 단순한 트리가 더 잘 일반화됨

 [최종 모델 설정]
  - LightGBM: learning_rate=0.0323, num_leaves=32, colsample_bytree=0.466
              reg_alpha=3.546, reg_lambda=2.186, n_estimators=3000
  - CatBoost: iterations=2000, depth=6, learning_rate=0.05
  - 앙상블: 가중 평균 (OOF AUC 기반 자동 계산)
==============================================================================
"""

In [ ]:
import koreanize_matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import MultiLabelBinarizer
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False


df_raw      = pd.read_csv('train.csv')
df_test_raw = pd.read_csv('test.csv')
df_raw.columns      = [c.strip() for c in df_raw.columns]
df_test_raw.columns = [c.strip() for c in df_test_raw.columns]
TARGET = '임신 성공 여부'

COLS_TO_DROP = [
    'ID', '임신 시도 또는 마지막 임신 경과 연수', '난자 해동 경과일',
    '불임 원인 - 여성 요인', '불임 원인 - 자궁경부 문제',
    '불임 원인 - 정자 면역학적 요인', '불임 원인 - 정자 운동성',
    '불임 원인 - 정자 농도', '불임 원인 - 정자 형태',
    'PGS 시술 여부', 'PGD 시술 여부', '착상 전 유전 검사 사용 여부',
]

df      = df_raw.drop(columns=COLS_TO_DROP, errors='ignore').copy()
df_test = df_test_raw.drop(columns=COLS_TO_DROP, errors='ignore').copy()

AGE_INFO = {
    '만18-34세': {'median': 26.0,  'risk': 'Normal'},
    '만35-37세': {'median': 36.0,  'risk': 'High_Early'},
    '만38-39세': {'median': 38.5,  'risk': 'High_Early'},
    '만40-42세': {'median': 41.0,  'risk': 'High_Extreme'},
    '만43-44세': {'median': 43.5,  'risk': 'High_Extreme'},
    '만45-50세': {'median': 47.5,  'risk': 'High_Extreme'},
    '알 수 없음': {'median': None,  'risk': 'Unknown'},
}

MULTILABEL_COL = '배아 생성 주요 이유'
KNOWN_REASONS  = ['현재 시술용', '배아 저장용', '난자 저장용', '기증용', '연구용']


# =============================================================================
# STEP 1. 기증 난자 x 나이 교호작용 피처 추가
# =============================================================================
# [근거] 만45-50세 성공률(0.168) > 만43-44세(0.118) 역전 현상
#   - 원인: 고령일수록 기증 난자 사용 비율 급증
#   - 기증 난자 = 젊은 공여자 난자 = 본인 나이와 무관하게 착상률 높음
#   - 이 두 변수의 교호작용을 모델이 명시적으로 학습해야 역전 현상을 올바르게 해석

def verify_donor_age_interaction(df_raw):
    """기증 난자 × 나이 역전 현상 확인"""
    print("=" * 55)
    print("[검증] 나이별 기증 난자 사용 비율")
    print("=" * 55)
    donor_rate = df_raw.groupby('시술 당시 나이')['난자 출처'].apply(
        lambda x: (x == '기증').mean()
    ).sort_values(ascending=False)
    print(donor_rate.round(3).to_string())

    print("\n[검증] 기증 난자 사용 여부별 성공률")
    print(df_raw.groupby('난자 출처')[TARGET].mean().round(3).to_string())
    print("\n=> 만45-50세에서 기증 난자 비율이 높으면 역전 현상 설명 가능")

verify_donor_age_interaction(df_raw)


# =============================================================================
# STEP 2. 피처 엔지니어링 (기증 난자 교호작용 추가)
# =============================================================================
def engineer_features(df, df_raw_ref=None):
    df = df.copy()

    # --- 나이 피처 (팀원 코드) ---
    if '시술 당시 나이' in df.columns:
        df['Age_Median']     = df['시술 당시 나이'].map(
            lambda x: AGE_INFO.get(x, {'median': None})['median'])
        df['Age_Risk_Group'] = df['시술 당시 나이'].map(
            lambda x: AGE_INFO.get(x, {'risk': 'Unknown'})['risk'])
        df = df.drop(columns=['시술 당시 나이'])

    # --- 유전 검사 ---
    if df_raw_ref is not None:
        df['유전검사_시행여부'] = (
            df_raw_ref['PGS 시술 여부'].notna() |
            df_raw_ref['PGD 시술 여부'].notna()
        ).astype(int).values

    # --- 기증 난자 × 나이 교호작용 (신규) ---
    # 기증 난자 여부 이진화
    if '난자 출처' in df.columns:
        df['기증난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)

        # 핵심 교호작용: 기증 난자이면 나이 패널티 제거
        # Age_Median이 높아도 기증 난자면 성공률이 높음 -> 이 관계를 직접 표현
        age_median = df['Age_Median'].fillna(47.5)
        df['기증난자×나이'] = df['기증난자_여부'] * age_median
        # 기증 난자가 아닐 때의 나이 효과만 분리
        df['자가난자×나이'] = (1 - df['기증난자_여부']) * age_median

    # --- 배아 품질 지표 ---
    total    = df.get('총 생성 배아 수', pd.Series(0, index=df.index))
    transfer = df.get('이식된 배아 수',  pd.Series(0, index=df.index))
    df['배아_잉여율']  = np.where(total > 0, (total - transfer) / total, 0).clip(0, 1)
    df['Age×배아수']  = df['Age_Median'].fillna(47.5) * total

    collected = df.get('수집된 신선 난자 수', pd.Series(0, index=df.index))
    mixed     = df.get('혼합된 난자 수',      pd.Series(0, index=df.index))
    df['수정률'] = np.where(collected > 0, mixed / collected, 0).clip(0, 1)

    stored = df.get('저장된 배아 수', pd.Series(0, index=df.index))
    df['저장배아_보유여부'] = (stored > 0).astype(int)

    def parse_count(col_name):
        s = df.get(col_name, pd.Series('0회', index=df.index))
        return s.astype(str).str.extract(r'(\d+)')[0].fillna(0).astype(int)

    df['총_시술횟수']    = parse_count('IVF 시술 횟수') + parse_count('DI 시술 횟수')
    df['기증정자_사용여부'] = df.get(
        '기증자 정자와 혼합된 난자 수',
        pd.Series(np.nan, index=df.index)
    ).notna().astype(int)

    return df


def encode_multilabel(df):
    df = df.copy()
    if MULTILABEL_COL not in df.columns:
        return df
    split_series = df[MULTILABEL_COL].fillna('').apply(
        lambda x: [v.strip() for v in x.split(',') if v.strip()]
    )
    mlb = MultiLabelBinarizer(classes=KNOWN_REASONS)
    encoded = pd.DataFrame(
        mlb.fit_transform(split_series),
        columns=[f'목적_{c}' for c in KNOWN_REASONS],
        index=df.index
    )
    return pd.concat([df.drop(columns=[MULTILABEL_COL]), encoded], axis=1)


def preprocess_missing(df, target=None):
    df = df.copy()
    if target and target in df.columns:
        y = df.pop(target)
    else:
        y = None
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    num_cols = df.select_dtypes(exclude='object').columns.tolist()
    df[cat_cols] = df[cat_cols].fillna('알 수 없음')
    df[num_cols] = df[num_cols].fillna(0)
    if y is not None:
        df[target] = y.values
    return df


def build_features(df, df_raw_ref=None, target=None):
    df = engineer_features(df, df_raw_ref=df_raw_ref)
    df = encode_multilabel(df)
    df = preprocess_missing(df, target=target)
    cat_cols = [c for c in df.select_dtypes(include='object').columns if c != target]
    for col in cat_cols:
        df[col] = df[col].astype('category')
    if target and target in df.columns:
        return df.drop(columns=[target]), df[target]
    return df, None


print("\n[피처 생성 중...]")
X, y      = build_features(df,      df_raw_ref=df_raw,      target=TARGET)
X_test, _ = build_features(df_test, df_raw_ref=df_test_raw, target=None)
for col in set(X.columns) - set(X_test.columns):
    X_test[col] = 0
X_test = X_test[X.columns]
print(f"최종 피처 수: {X.shape[1]}개")
print(f"신규 피처: 기증난자_여부, 기증난자×나이, 자가난자×나이")


# =============================================================================
# STEP 3. Optuna 하이퍼파라미터 최적화
# =============================================================================
# [왜 지금 Optuna인가?]
# best_iter=200~270 -> 아직 학습 여지 있음
# 현재 num_leaves=127, lr=0.02는 수동 추측값
# Optuna가 50회 탐색으로 AUC 0.003~0.005 추가 개선 가능

def run_optuna(X, y, n_trials=50, cv_splits=3):
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    scale_pos_weight = (y == 0).sum() / (y == 1).sum()
    cat_feats = [c for c in X.columns if X[c].dtype.name == 'category']
    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)

    def objective(trial):
        params = {
            'objective':         'binary',
            'metric':            'auc',
            'n_estimators':      3000,
            'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
            'subsample_freq':    1,
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 1.0),
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'scale_pos_weight':  scale_pos_weight,
            'random_state':      42,
            'n_jobs':            -1,
            'verbose':           -1,
        }
        aucs = []
        for tr_idx, val_idx in skf.split(X, y):
            model = lgb.LGBMClassifier(**params)
            model.fit(
                X.iloc[tr_idx], y.iloc[tr_idx],
                eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
                callbacks=[lgb.early_stopping(50, verbose=False),
                           lgb.log_evaluation(-1)],
                categorical_feature=cat_feats,
            )
            aucs.append(roc_auc_score(y.iloc[val_idx],
                                      model.predict_proba(X.iloc[val_idx])[:, 1]))
        return np.mean(aucs)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    print(f"[Optuna] {n_trials}회 탐색 시작...")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print(f"\n[Optuna 완료] Best 3-fold AUC: {study.best_value:.4f}")
    print("[Best 파라미터]")
    for k, v in study.best_params.items():
        print(f"  {k:25s}: {v}")
    return study.best_params


# =============================================================================
# STEP 4. LightGBM OOF 학습 함수
# =============================================================================
def run_lgbm_oof(X, y, X_test, params, n_splits=5, tag='lgbm'):
    scale_pos_weight = (y == 0).sum() / (y == 1).sum()
    params = {**params, 'scale_pos_weight': scale_pos_weight,
              'objective': 'binary', 'metric': 'auc',
              'random_state': 42, 'n_jobs': -1, 'verbose': -1}

    skf        = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_preds  = np.zeros(len(y))
    test_preds = np.zeros(len(X_test))
    cat_feats  = [c for c in X.columns if X[c].dtype.name == 'category']
    models, fold_aucs = [], []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
            callbacks=[lgb.early_stopping(100, verbose=False),
                       lgb.log_evaluation(500)],
            categorical_feature=cat_feats,
        )
        oof_preds[val_idx]  = model.predict_proba(X.iloc[val_idx])[:, 1]
        test_preds         += model.predict_proba(X_test)[:, 1] / n_splits
        auc = roc_auc_score(y.iloc[val_idx], oof_preds[val_idx])
        fold_aucs.append(auc)
        print(f"  [{tag}] Fold {fold+1}: AUC={auc:.4f}  iter={model.best_iteration_}")
        models.append(model)

    oof_auc = roc_auc_score(y, oof_preds)
    print(f"\n  [{tag}] OOF AUC={oof_auc:.4f}  (vs 0.7392: {oof_auc-0.7392:+.4f})")
    return models, oof_preds, test_preds, oof_auc


# =============================================================================
# STEP 5. CatBoost OOF 학습 함수
# =============================================================================
# [왜 CatBoost인가?]
# LightGBM: 범주형을 label encoding으로 변환 후 처리
# CatBoost: 범주형을 target statistics로 내부 처리 -> 범주형 많은 의료 데이터에 강점
# 두 모델은 오류 패턴이 달라서 앙상블 시 서로의 약점을 보완

def run_catboost_oof(X, y, X_test, n_splits=5):
    try:
        from catboost import CatBoostClassifier
    except ImportError:
        print("!pip install catboost 후 재실행하세요.")
        return None, None, None, None

    scale_pos_weight = (y == 0).sum() / (y == 1).sum()
    cat_feats_idx = [i for i, c in enumerate(X.columns)
                     if X[c].dtype.name == 'category']

    # CatBoost는 category dtype 대신 문자열 원본을 선호
    X_cb      = X.copy()
    X_test_cb = X_test.copy()
    for col in X.columns:
        if X[col].dtype.name == 'category':
            X_cb[col]      = X_cb[col].astype(str)
            X_test_cb[col] = X_test_cb[col].astype(str)

    params = {
        'iterations':        2000,
        'learning_rate':     0.05,
        'depth':             6,
        'l2_leaf_reg':       3,
        'scale_pos_weight':  scale_pos_weight,
        'eval_metric':       'AUC',
        'random_seed':       42,
        'verbose':           False,
        'early_stopping_rounds': 100,
    }

    skf        = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_preds  = np.zeros(len(y))
    test_preds = np.zeros(len(X_test))
    models, fold_aucs = [], []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cb, y)):
        model = CatBoostClassifier(**params)
        model.fit(
            X_cb.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=(X_cb.iloc[val_idx], y.iloc[val_idx]),
            cat_features=cat_feats_idx,
        )
        oof_preds[val_idx]  = model.predict_proba(X_cb.iloc[val_idx])[:, 1]
        test_preds         += model.predict_proba(X_test_cb)[:, 1] / n_splits
        auc = roc_auc_score(y.iloc[val_idx], oof_preds[val_idx])
        fold_aucs.append(auc)
        print(f"  [catboost] Fold {fold+1}: AUC={auc:.4f}")
        models.append(model)

    oof_auc = roc_auc_score(y, oof_preds)
    print(f"\n  [catboost] OOF AUC={oof_auc:.4f}  (vs 0.7392: {oof_auc-0.7392:+.4f})")
    return models, oof_preds, test_preds, oof_auc


# =============================================================================
# STEP 6. 가중 앙상블
# =============================================================================
def weighted_ensemble(oof_list, test_list, y, weights=None):
    """
    OOF 기반 최적 가중치 탐색 후 앙상블.
    weights=None이면 AUC 기반 자동 계산.
    """
    if weights is None:
        # 각 모델의 OOF AUC를 가중치로 사용
        aucs = [roc_auc_score(y, oof) for oof in oof_list]
        total = sum(aucs)
        weights = [a / total for a in aucs]
        print(f"[앙상블] 자동 가중치: {[f'{w:.3f}' for w in weights]}")

    oof_ensemble  = sum(w * o for w, o in zip(weights, oof_list))
    test_ensemble = sum(w * t for w, t in zip(weights, test_list))
    ens_auc = roc_auc_score(y, oof_ensemble)
    print(f"[앙상블] OOF AUC={ens_auc:.4f}  (vs 0.7392: {ens_auc-0.7392:+.4f})")
    return oof_ensemble, test_ensemble


In [ ]:
#  LightGBM (기본 파라미터로 먼저 확인) ---
lgbm_base_params = {
    'n_estimators': 3000, 'learning_rate': 0.02,
    'num_leaves': 127, 'min_child_samples': 30,
    'subsample': 0.8, 'subsample_freq': 1,
    'colsample_bytree': 0.7, 'reg_alpha': 0.1, 'reg_lambda': 2.0,
}
lgbm_models, lgbm_oof, lgbm_test, lgbm_auc = run_lgbm_oof(
    X, y, X_test, lgbm_base_params, tag='lgbm_base')





In [ ]:
# 수정 코드

submission = pd.read_csv('sample_submission.csv')

# 컬럼 구조 먼저 확인
print("sample_submission 컬럼:", submission.columns.tolist())
# 출력: ['ID', 'probability']

# 'probability' 컬럼에 예측값 채우기
submission['probability'] = lgbm_test

# 최종 확인
print(submission.head(3))
print(f"컬럼 수: {len(submission.columns)}개  |  행 수: {len(submission)}")

submission.to_csv('submission_v4.csv', index=False)
print("저장 완료")

In [ ]:
# --- 실행 2: Optuna 최적화
best_params = run_optuna(X, y, n_trials=50, cv_splits=3)
lgbm_opt_params = {**best_params, 'n_estimators': 3000}
lgbm_opt_models, lgbm_opt_oof, lgbm_opt_test, lgbm_opt_auc = run_lgbm_oof(
    X, y, X_test, lgbm_opt_params, tag='lgbm_optuna')

# Optuna 결과 저장
submission = pd.read_csv('sample_submission.csv')
submission['probability'] = lgbm_opt_test
submission.to_csv('submission_optuna.csv', index=False)
print(f"저장 완료: submission_optuna.csv  |  OOF AUC: {lgbm_opt_auc:.4f}")

In [ ]:
# Optuna 결과 저장
submission = pd.read_csv('sample_submission.csv')
submission['probability'] = lgbm_opt_test
submission.to_csv('submission_optuna.csv', index=False)
print(f"저장 완료: submission_optuna.csv  |  OOF AUC: {lgbm_opt_auc:.4f}")

In [ ]:
# !pip install catboost

In [ ]:
# --- 실행 3: CatBoost
from catboost import CatBoostClassifier

cat_models, cat_oof, cat_test, cat_auc = run_catboost_oof(X, y, X_test)

In [ ]:
# CatBoost 결과 저장
submission = pd.read_csv('sample_submission.csv')
submission['probability'] = cat_test
submission.to_csv('submission_catboost.csv', index=False)
print(f"저장 완료: submission_catboost.csv  |  OOF AUC: {cat_auc:.4f}")

In [ ]:
# 바로 앙상블도 실행
oof_ens, test_ens = weighted_ensemble(
    oof_list=[lgbm_opt_oof, cat_oof],
    test_list=[lgbm_opt_test, cat_test],
    y=y
)

submission = pd.read_csv('sample_submission.csv')
submission['probability'] = test_ens
submission.to_csv('submission_ensemble.csv', index=False)
print(f"앙상블 저장 완료")